# Parallel and majority voting

This notebook covers Chapter 7 section 7.2.1: helpers that aren't full modules but help you build more reliable programs. We'll cover:

- `dspy.majority` — self-consistency via majority voting across multiple completions.
- `dspy.Parallel` — run a list of `(module, input)` pairs concurrently.
- `module.batch()` — the simpler one-module-many-inputs version of `Parallel`.

**Environment variables**: `OPENAI_API_KEY` (loaded from `.env`).


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))


## `dspy.majority` for self-consistency

Generate N completions with `ChainOfThought`, then take a majority vote.


In [ ]:
cot = dspy.ChainOfThought("entity: str -> entity_type: str")

# Generate 5 completions
result = cot(entity="Apple", config={"n": 5, "temperature": 0.9})

# Take majority vote
final = dspy.majority(result.completions)
print(final.entity_type)  # Most common answer across 5 attempts


### Custom normalization

For fuzzy matching across synonyms, pass your own `normalize` function.


In [ ]:
def normalize_location(text):
    aliases = {"nyc": "new york city", "la": "los angeles", "sf": "san francisco"}
    return aliases.get(text.strip().lower(), text.strip().lower())

final = dspy.majority(result.completions, normalize=normalize_location)


## `dspy.Parallel` for concurrent execution

Give `Parallel` a list of `(module, input)` pairs and it runs them all concurrently. Total cost is the same as running sequentially, but the wall-clock time drops sharply.


In [ ]:
import dspy

translate = dspy.ChainOfThought(
    "text: str, target_language: str -> translation: str"
)

product_desc = "Wireless noise-cancelling headphones with 30-hour battery life"
languages = ["Spanish", "French", "German", "Japanese", "Portuguese"]

# Build execution pairs
exec_pairs = [
    (translate, dspy.Example(text=product_desc, target_language=lang).with_inputs("text", "target_language"))
    for lang in languages
]

# Run all 5 translations concurrently
parallel = dspy.Parallel(num_threads=5, max_errors=2)
results = parallel(exec_pairs)

for lang, result in zip(languages, results):
    print(f"{lang}: {result.translation}")


## `module.batch()` for the simpler case

When you're running the same module over many inputs, `module.batch()` is the lightweight equivalent of `Parallel`.


In [ ]:
# Equivalent to Parallel for single-module batch processing
examples = [dspy.Example(text=product_desc, target_language=lang).with_inputs("text", "target_language") for lang in languages]
results = translate.batch(examples, num_threads=5)
